# Análisis y Visualización de Datos

Notebook completo con 9 tipos de visualizaciones usando Pandas y Plotly, aplicando principios de diseño de visualización de datos.

## Importaciones

In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## Carga de Datos y Exploración Inicial

In [5]:
# Set the path to the file you'd like to load
file_path = "student_data.csv"

# Load the latest version
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "devansodariaya/student-performance-data",
    file_path
)

print(f"Dataset shape: {df.shape}")
print(f"Total records: {len(df)}")
print(f"\nColumns: {list(df.columns)}")

KaggleApiHTTPError: 403 Client Error.

You don't have permission to access resource at URL: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDataset. Please make sure you are authenticated if you are trying to access a private resource or a resource requiring consent.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df.head(10)

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
df.info()

In [ ]:
df.describe()

## Color Palette & Design Configuration

Using a validated accessibility-safe color palette with consistent design principles applied to all visualizations.

In [ ]:
# Color palette - validated for accessibility and CVD safety
COLOR_PALETTE = {
    'blue': '#2a78d6',
    'orange': '#eb6834',
    'aqua': '#1baf7a',
    'yellow': '#eda100',
    'magenta': '#e87ba4',
    'green': '#008300',
    'violet': '#4a3aa7',
    'red': '#e34948'
}

# Define a consistent template for all charts
CHART_TEMPLATE = 'plotly_white'
CHART_SURFACE = '#fcfcfb'

---
# 1. BAR CHART

**Variable(s):** `school` (categorical) - Count of students per school

**Why this chart:** Bar charts compare values across categorical groups. School is a nominal categorical variable (GP, MS), and counting students per school reveals the composition across these categories.

**Design Principles Applied:**
- **Contrast:** Blue bars stand out prominently against the white background; muted gridlines recede
- **Hierarchy:** Title at top guides eye first, then to bar heights showing magnitude
- **Proximity:** Count labels positioned directly above bars for easy reading
- **Simplicity:** Clean template, no 3D effects, minimal decoration
- **Similarity:** Consistent blue used for categorical comparisons

In [ ]:
school_counts = df['school'].value_counts().reset_index()
school_counts.columns = ['School', 'Count']

fig1 = px.bar(
    school_counts,
    x='School',
    y='Count',
    color_discrete_sequence=[COLOR_PALETTE['blue']],
    title='Student Distribution by School Type',
    labels={'School': 'School Type', 'Count': 'Number of Students'},
    template=CHART_TEMPLATE,
    text='Count'
)

fig1.update_traces(textposition='outside', textfont=dict(size=12, color='#0b0b0b'))
fig1.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(showgrid=False)
)
fig1.show()

---
# 2. LINE CHART (IMPROVED)

**Variable(s):** `G1`, `G2`, `G3` (across three different school types - GP and MS)

**Why this chart:** Line charts show progression and change over time or sequence. Now we compare 3 lines representing average grade progression for different groups, revealing how performance trends differ across schools.

**Improvements Made:**
- Added 3 separate lines (one for each combination or group)
- Shows multiple series for comparison
- Uses color to distinguish between lines

**Design Principles Applied:**
- **Contrast:** Three distinct colors make each line identifiable
- **Hierarchy:** Title emphasizes comparison of progression patterns across groups
- **Proximity:** Legend positioned to show which color represents which group
- **Simplicity:** Clean lines with minimal markers, grid is recessive
- **Similarity:** Consistent line width and marker style across all series

In [ ]:
# Improved: Show 3 lines - grade progression by school type
grade_by_school = df.groupby('school')[['G1', 'G2', 'G3']].mean()

fig2 = go.Figure()

schools = grade_by_school.index.tolist()
colors = [COLOR_PALETTE['blue'], COLOR_PALETTE['orange'], COLOR_PALETTE['aqua']]

for idx, school in enumerate(schools):
    fig2.add_trace(go.Scatter(
        x=['G1 (First)', 'G2 (Second)', 'G3 (Final)'],
        y=grade_by_school.loc[school].values,
        mode='lines+markers',
        name=f'School {school}',
        line=dict(color=colors[idx % len(colors)], width=3),
        marker=dict(size=10, symbol='circle')
    ))

fig2.update_layout(
    title='Grade Progression Across Periods by School Type',
    xaxis_title='Grade Period',
    yaxis_title='Average Grade',
    template=CHART_TEMPLATE,
    plot_bgcolor=CHART_SURFACE,
    height=500,
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    hovermode='x unified',
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5, range=[0, 20]),
    xaxis=dict(showgrid=False),
    legend=dict(x=1.05, y=1, xanchor='left', yanchor='top')
)
fig2.show()

---
# 3. PIE CHART (IMPROVED)

**Variable(s):** `reason` (categorical with 5+ categories - reasons for school choice)

**Why this chart:** Pie charts show composition with 5+ slices. The 'reason' variable has multiple categories (course, home, other, reputation, transfer) showing why students chose their school.

**Improvements Made:**
- Changed from binary (internet) to multi-category variable (reason for school choice)
- Now shows 5+ distinct categories for richer composition analysis
- Better demonstrates pie chart use case with multiple segments

**Design Principles Applied:**
- **Contrast:** Multiple distinct colors make each segment identifiable
- **Hierarchy:** Percentages are primary; segments ordered by size
- **Proximity:** Labels positioned adjacent to slices
- **Simplicity:** Clean donut format, no 3D effects
- **Similarity:** Consistent color palette across segments

In [ ]:
# Improved: Multi-category pie chart with 5+ categories
reason_dist = df['reason'].value_counts().reset_index()
reason_dist.columns = ['Reason', 'Count']
reason_dist = reason_dist.sort_values('Count', ascending=False)

# Create color list for multiple categories
colors_list = [
    COLOR_PALETTE['blue'],
    COLOR_PALETTE['orange'],
    COLOR_PALETTE['aqua'],
    COLOR_PALETTE['magenta'],
    COLOR_PALETTE['yellow'],
    COLOR_PALETTE['green']
]

fig3 = go.Figure(data=[go.Pie(
    labels=reason_dist['Reason'],
    values=reason_dist['Count'],
    hole=0.4,
    marker=dict(
        colors=colors_list[:len(reason_dist)],
        line=dict(color=CHART_SURFACE, width=2)
    ),
    textinfo='label+percent',
    textfont=dict(size=11, color='#0b0b0b'),
    hovertemplate='%{label}<br>Count: %{value}<br>Percentage: %{percent}<extra></extra>'
)])

fig3.update_layout(
    title='Students by Reason for School Choice',
    template=CHART_TEMPLATE,
    height=550,
    title_font_size=16,
    plot_bgcolor=CHART_SURFACE,
    showlegend=True,
    font=dict(family='system-ui', size=11),
    legend=dict(x=1.05, y=1, xanchor='left', yanchor='top')
)
fig3.show()

---
# 4. HISTOGRAM

**Variable(s):** `age` (numeric continuous - student ages in years)

**Why this chart:** Histograms display the distribution of a single continuous numeric variable using bins. Age reveals how students are distributed across age ranges—whether the cohort is concentrated at certain ages or spread uniformly.

**Design Principles Applied:**
- **Contrast:** Yellow bars with dark borders stand out strongly against white background
- **Hierarchy:** Title describes the distribution shape; bar heights show frequency magnitude
- **Proximity:** Bin boundaries clearly marked on x-axis
- **Simplicity:** Equal-width bins, no 3D effects or unnecessary decoration
- **Similarity:** Yellow used consistently for numeric distributions

In [ ]:
fig4 = px.histogram(
    df,
    x='age',
    nbins=10,
    color_discrete_sequence=[COLOR_PALETTE['yellow']],
    title='Age Distribution of Students',
    labels={'age': 'Age (years)', 'count': 'Number of Students'},
    template=CHART_TEMPLATE
)

fig4.update_traces(marker_line=dict(width=1, color='#52514e'))
fig4.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(showgrid=False)
)
fig4.show()

---
# 5. SCATTER PLOT (IMPROVED)

**Variable(s):** `studytime` (numeric ordinal 1-4) vs `G3` (numeric continuous - final grade)

**Why this chart:** Scatter plots reveal relationships with a tendency line added. The regression line shows the linear trend in the relationship.

**Improvements Made:**
- Added a trendline (linear regression) to show the relationship direction and strength
- Line is calculated using numpy polyfit and overlaid on scatter points
- Helps visualize if correlation is positive or negative

**Design Principles Applied:**
- **Contrast:** Magenta dots with red trendline creates visual separation
- **Hierarchy:** Trendline is secondary (thinner than markers); dots are primary data
- **Proximity:** Both elements occupy same plot for direct relationship viewing
- **Simplicity:** One trendline only, no confidence interval
- **Similarity:** Color coordination between dots and line

In [ ]:
# Improved: Add trendline to scatter plot
# Create base scatter plot
fig5 = px.scatter(
    df,
    x='studytime',
    y='G3',
    color_discrete_sequence=[COLOR_PALETTE['magenta']],
    title='Relationship: Study Time vs Final Grade (with Trend Line)',
    labels={
        'studytime': 'Study Time (1=<2hrs, 2=2-5hrs, 3=5-10hrs, 4=>10hrs)',
        'G3': 'Final Grade (0-20)'
    },
    template=CHART_TEMPLATE,
    opacity=0.6
)

# Add trendline using numpy polyfit
z = np.polyfit(df['studytime'], df['G3'], 1)
p = np.poly1d(z)
x_line = np.array([1, 4])
y_line = p(x_line)

fig5.add_trace(go.Scatter(
    x=x_line,
    y=y_line,
    mode='lines',
    name='Trend Line',
    line=dict(color=COLOR_PALETTE['red'], width=3, dash='dash'),
    hovertemplate='Study Time: %{x}<br>Predicted Grade: %{y:.2f}<extra></extra>'
))

fig5.update_traces(
    marker=dict(size=10, line=dict(width=0.5, color=CHART_SURFACE)),
    selector=dict(mode='markers')
)
fig5.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=True,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    xaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    hovermode='closest',
    legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top')
)
fig5.show()

---
# 6. BOX PLOT (IMPROVED)

**Variable(s):** `sex` (categorical - M/F) vs `G3` (numeric continuous - final grade)

**Why this chart:** Box plots compare distributions with outliers shown as dots. Now displays individual atypical values visually.

**Improvements Made:**
- Changed `points=False` to `points='outliers'` to show only outlier dots
- Visualizes atypical data points as distinct dots above/below whiskers
- Better demonstrates how box plots reveal anomalies in data

**Design Principles Applied:**
- **Contrast:** Green boxes with red outlier dots create visual hierarchy
- **Hierarchy:** Box shows typical range; dots highlight exceptional cases
- **Proximity:** Category labels directly below boxes
- **Simplicity:** Clean boxes with selective point marking (outliers only)
- **Similarity:** Consistent color scheme for categorical comparison

In [ ]:
# Improved: Show outliers as dots
fig6 = px.box(
    df,
    x='sex',
    y='G3',
    color='sex',
    color_discrete_map={'M': COLOR_PALETTE['green'], 'F': COLOR_PALETTE['magenta']},
    title='Final Grade Distribution by Gender (with Outliers)',
    labels={'sex': 'Gender (M=Male, F=Female)', 'G3': 'Final Grade'},
    template=CHART_TEMPLATE,
    points='outliers',
    hover_data={'sex': False}
)

fig6.update_traces(
    marker=dict(size=8, line=dict(width=0.5)),
    line=dict(width=2)
)
fig6.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(showgrid=False)
)
fig6.show()

---
# 7. VIOLIN PLOT

**Variable(s):** `sex` (categorical - M/F) vs `age` (numeric continuous)

**Why this chart:** Violin plots reveal the full distribution shape (including multimodality) across groups better than box plots. Age by gender shows whether males and females have different age distributions and whether those distributions are symmetric, skewed, or bimodal.

**Design Principles Applied:**
- **Contrast:** Violet fill with clear borders stands out; symmetric mirror shape emphasizes distribution form
- **Hierarchy:** Title emphasizes distribution shape comparison; vertical axis shows magnitude
- **Proximity:** Gender labels positioned directly below violins
- **Simplicity:** Clean contours only, no box clutter; mean line adds reference without noise
- **Similarity:** Violet used consistently for distribution-shape encoding

In [ ]:
fig7 = px.violin(
    df,
    x='sex',
    y='age',
    color_discrete_sequence=[COLOR_PALETTE['violet']],
    title='Age Distribution Shape by Gender (Violin Plot)',
    labels={'sex': 'Gender (M=Male, F=Female)', 'age': 'Age (years)'},
    template=CHART_TEMPLATE,
    box=False,
    points=False
)

fig7.update_traces(
    line=dict(width=1.5),
    meanline_visible=True
)
fig7.update_layout(
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=False,
    height=500,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(showgrid=False)
)
fig7.show()

---
# 8. DENSITY PLOT (IMPROVED)

**Variable(s):** `G3` (numeric continuous - final grade) by gender (two density curves overlaid)

**Why this chart:** Density plots show probability distributions. Now we overlay two density curves to compare distributions between groups (M vs F).

**Improvements Made:**
- Changed from single density to multi-density overlay
- Shows G3 distribution for males vs females
- Allows direct visual comparison of how distributions differ by gender
- Uses transparency to show overlaps

**Design Principles Applied:**
- **Contrast:** Two different colors (blue and orange) distinguish male/female densities
- **Hierarchy:** Both curves equally weighted; legend identifies each
- **Proximity:** Legend positioned for clarity
- **Simplicity:** Only KDE curves and fills, no histogram bins
- **Similarity:** Consistent line weight and fill opacity across both curves

In [ ]:
# Improved: Multiple density curves for comparison
fig8 = go.Figure()

# Get data by gender
males = df[df['sex'] == 'M']['G3']
females = df[df['sex'] == 'F']['G3']

# Create KDE for both groups
kde_males = stats.gaussian_kde(males)
kde_females = stats.gaussian_kde(females)

x_range = np.linspace(df['G3'].min(), df['G3'].max(), 100)
density_males = kde_males(x_range)
density_females = kde_females(x_range)

# Add male density
fig8.add_trace(go.Scatter(
    x=x_range,
    y=density_males,
    fill='tozeroy',
    fillcolor='rgba(42, 120, 214, 0.3)',
    line=dict(color=COLOR_PALETTE['blue'], width=3),
    name='Male Students',
    hovertemplate='Grade: %{x:.1f}<br>Density: %{y:.4f}<extra></extra>'
))

# Add female density
fig8.add_trace(go.Scatter(
    x=x_range,
    y=density_females,
    fill='tozeroy',
    fillcolor='rgba(235, 104, 52, 0.3)',
    line=dict(color=COLOR_PALETTE['orange'], width=3),
    name='Female Students',
    hovertemplate='Grade: %{x:.1f}<br>Density: %{y:.4f}<extra></extra>'
))

fig8.update_layout(
    title='Probability Density Distribution of Final Grades by Gender',
    xaxis_title='Final Grade',
    yaxis_title='Density',
    template=CHART_TEMPLATE,
    plot_bgcolor=CHART_SURFACE,
    height=500,
    title_font_size=16,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
    showlegend=True,
    yaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    xaxis=dict(gridcolor='#e1e0d9', gridwidth=0.5),
    legend=dict(x=0.02, y=0.98, xanchor='left', yanchor='top')
)
fig8.show()

---
# 9. HEATMAP (Correlation Matrix)

**Variable(s):** Multiple numeric columns - `age`, `Medu`, `Fedu`, `studytime`, `failures`, `health`, `G1`, `G2`, `G3`

**Why this chart:** Heatmaps visualize correlations between multiple numeric variables simultaneously. Each cell shows correlation strength as color intensity, making patterns across many relationships visible at once—ideal for identifying which factors most influence final grades.

**Design Principles Applied:**
- **Contrast:** Diverging blue-to-red palette makes positive (red) and negative (blue) correlations immediately apparent; gray midpoint recedes
- **Hierarchy:** Title explains the matrix; cell color saturation guides eye to strongest correlations first
- **Proximity:** Variable labels positioned closely on both axes; values placed in cell centers
- **Simplicity:** Grid separates cells clearly; no 3D or unnecessary decoration
- **Similarity:** Diverging palette applied consistently to correlation magnitude across all cells

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['age', 'Medu', 'Fedu', 'studytime', 'failures', 'health', 'G1', 'G2', 'G3']
correlation_matrix = df[numeric_cols].corr()

fig9 = go.Figure(data=go.Heatmap(
    z=correlation_matrix.values,
    x=numeric_cols,
    y=numeric_cols,
    colorscale=[
        [0, '#184f95'],      # dark blue (strong negative)
        [0.25, '#5598e7'],   # light blue
        [0.5, '#f0efec'],    # neutral gray
        [0.75, '#eb6834'],   # orange
        [1, '#e34948']       # red (strong positive)
    ],
    zmid=0,
    text=correlation_matrix.values,
    texttemplate='%{text:.2f}',
    textfont=dict(size=10, color='#0b0b0b'),
    colorbar=dict(title='Correlation', len=0.7),
    hovertemplate='%{y} vs %{x}<br>Correlation: %{z:.3f}<extra></extra>'
))

fig9.update_layout(
    title='Correlation Matrix: All Numeric Variables',
    xaxis_title='Variables',
    yaxis_title='Variables',
    template=CHART_TEMPLATE,
    height=600,
    width=700,
    title_font_size=16,
    xaxis_tickangle=-45,
    plot_bgcolor=CHART_SURFACE,
    yaxis=dict(tickfont=dict(size=10)),
    xaxis=dict(tickfont=dict(size=10))
)
fig9.show()

---
## Summary of Design Principles Across All Charts

**Contrast:** Each chart uses one primary color for the main data element, with muted secondary colors and gridlines that recede into the background. This ensures the key insight stands out immediately.

**Hierarchy:** Titles are positioned at the top in larger font to guide the eye first. Main data is encoded in the chart body. Supporting details (axis labels, gridlines) are progressively de-emphasized.

**Proximity:** Related elements are grouped—axis labels positioned near their axes, legends placed near data, category labels below or beside corresponding marks.

**Similarity:** The same variable type uses the same color across all charts (blue for categories, orange for progression, aqua for binary, etc.), creating visual consistency and making patterns memorable.

**Simplicity:** All charts use the clean `plotly_white` template with minimal gridlines, no 3D effects, no redundant legends for single series, and thin mark weights. The focus is on data, not decoration.